# EduHarness — Full Pipeline (Optional All-Colab Compatibility)

**不推荐作为默认。** API 只放 Colab、渲染/打包在本地时，请用：

`notebooks/06_colab_api_oneshot.ipynb`

本 notebook 仍保留「整条在 Colab 跑完（含渲染）」的兼容路径。需先跑 `01_colab_setup.ipynb`，并会在 Colab 安装 ffmpeg / manim 等重依赖。


In [ ]:
from __future__ import annotations
import json, subprocess
from pathlib import Path

from educast.colab_session import ensure_api_key, load_session

session = load_session()
ensure_api_key()
REPO, RUN_DIR = Path(session["REPO"]), Path(session["RUN_DIR"])

subprocess.check_call(["apt-get", "-qq", "update"])
subprocess.check_call([
    "apt-get", "-qq", "install", "-y", "ffmpeg",
    "libcairo2-dev", "libpango1.0-dev", "libffi-dev", "pkg-config",
])
subprocess.check_call([
    __import__("sys").executable, "-m", "pip", "install", "-q", "-r",
    str(REPO / "requirements" / "render.txt"),
])

from educast.config import Config
from educast.pipeline import load_request, run

REQUEST_PATH = Path(session.get("REQUEST", REPO / "examples" / "sample_request.json"))
request = load_request(REQUEST_PATH)
notes = REPO / "examples" / "binary_notes.txt"
if "binary" in REQUEST_PATH.name and notes.is_file() and not request.source_text:
    request.source_text = notes.read_text(encoding="utf-8")

cfg = Config.from_env(request, run_dir=RUN_DIR / request.request_id)
cfg.max_review_rounds = min(cfg.max_review_rounds, 2)
result = run(cfg)
print(json.dumps(result, indent=2))


In [ ]:
# List bundle artifacts
from pathlib import Path
import json
env = json.loads(Path("/content/eduharness_colab_env.json").read_text())
from educast.pipeline import load_request
from educast.config import Config
request = load_request(Path(env["REQUEST"]))
cfg = Config.from_env(request, run_dir=Path(env["RUN_DIR"]) / request.request_id)
for p in sorted(cfg.bundle_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(cfg.bundle_dir), p.stat().st_size)
